<a href="https://colab.research.google.com/github/swati-singh24/LSTM-next-word-predictor-/blob/main/dl_day63(mini_project).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Step 1: Read the uploaded text file
file_path = '/content/1661-0.txt'

with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
    text = f.read()

In [3]:
import tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer


In [4]:
tokenizer = Tokenizer(num_words=4000, oov_token='<UNK>')
tokenizer.fit_on_texts([text])
len(tokenizer.word_index) #total unique words

8932

In [5]:
input_sequences = []
for sentence in text.split('\n'):
  #saare sectences ko sequence mein convert kar rhe
  tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]
  #sequences generate kar rhe hain using supervised learning
  for i in range(1,len(tokenized_sentence)):
    input_sequences.append(tokenized_sentence[:i+1])

In [6]:
input_sequences

[[146, 1],
 [146, 1, 2],
 [146, 1, 2, 1021],
 [146, 1, 2, 1021, 5],
 [146, 1, 2, 1021, 5, 129],
 [146, 1, 2, 1021, 5, 129, 35],
 [146, 1, 2, 1021, 5, 129, 35, 46],
 [146, 1, 2, 1021, 5, 129, 35, 46, 612],
 [146, 1, 2, 1021, 5, 129, 35, 46, 612, 2236],
 [146, 1, 2, 1021, 5, 129, 35, 46, 612, 2236, 2237],
 [31, 1022],
 [31, 1022, 16],
 [31, 1022, 16, 24],
 [31, 1022, 16, 24, 2],
 [31, 1022, 16, 24, 2, 276],
 [31, 1022, 16, 24, 2, 276, 5],
 [31, 1022, 16, 24, 2, 276, 5, 395],
 [31, 1022, 16, 24, 2, 276, 5, 395, 2238],
 [31, 1022, 16, 24, 2, 276, 5, 395, 2238, 22],
 [31, 1022, 16, 24, 2, 276, 5, 395, 2238, 22, 52],
 [31, 1022, 16, 24, 2, 276, 5, 395, 2238, 22, 52, 1677],
 [31, 1022, 16, 24, 2, 276, 5, 395, 2238, 22, 52, 1677, 3],
 [31, 1022, 16, 24, 2, 276, 5, 395, 2238, 22, 52, 1677, 3, 19],
 [573, 52],
 [573, 52, 3399],
 [573, 52, 3399, 3400],
 [573, 52, 3399, 3400, 14],
 [573, 52, 3399, 3400, 14, 76],
 [573, 52, 3399, 3400, 14, 76, 818],
 [573, 52, 3399, 3400, 14, 76, 818, 11],
 [573, 5

In [7]:
#finding the sequence or sentence whose length is max for padding
max_len = max([len(x) for x in input_sequences])
max_len

20

In [8]:
#now we will do padding
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input_sequences = pad_sequences(input_sequences, maxlen = max_len, padding='pre')
padded_input_sequences


array([[   0,    0,    0, ...,    0,  146,    1],
       [   0,    0,    0, ...,  146,    1,    2],
       [   0,    0,    0, ...,    1,    2, 1021],
       ...,
       [   0,    0,    0, ...,    4,  361,   84],
       [   0,    0,    0, ...,  361,   84,  359],
       [   0,    0,    0, ...,   84,  359, 1674]], dtype=int32)

In [9]:
#now we are creating X and y(dependent and independent)
X = padded_input_sequences[:,:-1]#saare column liya last wala column chodke coz that was the o/p
y = padded_input_sequences[:,-1]#sirf last column liya coz that is tye o/p

In [10]:
X.shape

(101619, 19)

In [11]:
y.shape

(101619,)

In [12]:
#now we will do the one hot encoding for y as we are taking this task as classification
from tensorflow.keras.utils import to_categorical
y = to_categorical(y,num_classes=4000)

In [13]:
y.shape

(101619, 4000)

In [14]:
#building the architecture of model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,Dropout,BatchNormalization
from tensorflow.keras.regularizers import l2

In [15]:
vocab_size = 4000

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=200, input_shape=(20,)))

# Layer 1
model.add(LSTM(256, return_sequences=True, kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(Dropout(0.3))

# Layer 2
model.add(LSTM(256, kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(Dropout(0.3))

# Output
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [16]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,          # Agar 3 epochs tak validation loss kam nahi hua toh training rok dega
    restore_best_weights=True
)

history = model.fit(
    X, y,
    epochs=40,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stopping]
)

Epoch 1/40
636/636 ━━━━━━━━━━━━━━━━━━━━ 18s 18ms/step - accuracy: 0.1027 - loss: 6.3066 - val_accuracy: 0.1134 - val_loss: 5.7973
Epoch 2/40
636/636 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.1542 - loss: 5.1516 - val_accuracy: 0.1358 - val_loss: 5.5465
Epoch 3/40
636/636 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.1689 - loss: 4.8725 - val_accuracy: 0.1461 - val_loss: 5.4796
Epoch 4/40
636/636 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.1799 - loss: 4.6840 - val_accuracy: 0.1410 - val_loss: 5.5488
Epoch 5/40
636/636 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.1901 - loss: 4.5082 - val_accuracy: 0.1417 - val_loss: 5.5827
Epoch 6/40
636/636 ━━━━━━━━━━━━━━━━━━━━ 21s 17ms/step - accuracy: 0.2003 - loss: 4.3605 - val_accuracy: 0.1511 - val_loss: 5.6140
Epoch 7/40
636/636 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.2087 - loss: 4.2204 - val_accuracy: 0.1452 - val_loss: 5.7874
Epoch 8/40
636/636 ━━━━━━━━━━━━━━━━━━━━ 20s 17ms/step - accuracy: 0.2189 - loss: 4.0981 - 

In [21]:
import time
import numpy as np
text = "what is your "

for i in range(10):
  # tokenize
  token_text = tokenizer.texts_to_sequences([text])[0]
  # padding
  padded_token_text = pad_sequences([token_text], maxlen=56, padding='pre')
  # predict
  pos = np.argmax(model.predict(padded_token_text))

  for word,index in tokenizer.word_index.items():
    if index == pos:
      text = text + " " + word
      print(text)
      time.sleep(2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
what is your  own
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
what is your  own and
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
what is your  own and i
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
what is your  own and i was
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
what is your  own and i was a
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
what is your  own and i was a few
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
what is your  own and i was a few minutes
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
what is your  own and i was a few minutes of
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
what is your  own and i was a few minutes of the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
what is your  own and i was a few minutes of the <UNK>
